# 15. Supervised Learning: Decision Trees

## Algorithm Category
**Type**: Supervised Learning - Classification/Regression  
**Complexity**: Medium  
**Use Case**: Tree-based decision making with interpretable rules

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand how decision trees make decisions through recursive splitting
- Implement decision trees for both classification and regression
- Understand entropy, Gini impurity, and information gain
- Visualize decision trees and interpret their structure
- Tune hyperparameters to prevent overfitting
- Apply decision trees to real-world problems

## Historical Context

Decision trees have roots in decision theory and were formalized in the 1960s. Key developments include:
- ID3 algorithm (Quinlan, 1986) - used information gain
- C4.5 algorithm (Quinlan, 1993) - improved handling of continuous features
- CART algorithm (Breiman et al., 1984) - Classification and Regression Trees

**Key Papers/References:**
- Breiman, L., et al. (1984). "Classification and Regression Trees"
- Quinlan, J.R. (1986). "Induction of Decision Trees"
- Quinlan, J.R. (1993). "C4.5: Programs for Machine Learning"

## When to Use Decision Trees

Decision trees are appropriate when:
- Interpretability is crucial (easy to visualize and explain)
- You need to understand feature importance
- Data has non-linear relationships
- You want a baseline before using ensemble methods
- Working with mixed data types (categorical and numerical)

## Theory & Mechanics

### Mathematical Foundation

Decision trees recursively partition the feature space by asking yes/no questions about feature values.

**Entropy (for classification):**
$$H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$

**Gini Impurity:**
$$Gini(S) = 1 - \sum_{i=1}^{c} p_i^2$$

**Information Gain:**
$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

**Variance Reduction (for regression):**
$$\text{Var}(S) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \bar{y})^2$$

### How It Works

1. **Start**: Begin with all training data at root node
2. **Split**: Find best feature and threshold that maximizes information gain (or minimizes impurity)
3. **Recurse**: Repeat for each child node until stopping criterion met
4. **Leaf**: Assign class (majority vote) or value (mean) to leaf nodes
5. **Prediction**: Traverse tree from root to leaf based on feature values

### Key Hyperparameters

- **max_depth**: Maximum depth of tree (prevents overfitting)
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples required in a leaf node
- **max_features**: Number of features to consider for best split
- **criterion**: Splitting criterion ('gini', 'entropy' for classification; 'mse', 'mae' for regression)

### Limitations

- Prone to overfitting (high variance)
- Sensitive to small changes in data
- Can create biased trees if classes are imbalanced
- May not capture linear relationships efficiently


## Implementation

Let's implement decision trees for both classification and regression.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_diabetes
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error

# Import our helper functions
from src.models.supervised import split_data, evaluate_classifier, evaluate_regressor
from src.models.classification import calculate_classification_metrics, plot_confusion_matrix
from src.utils.benchmarking import benchmark_model_training
from src.utils.traceability import extract_feature_importance_trace, trace_decision_path, save_traceability_data
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Classification Example: Iris dataset
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='Species')

print(f"Dataset Shape: {X.shape}")
print(f"Classes: {iris.target_names.tolist()}")
print(f"Class distribution:\n{y.value_counts()}")

# Split data
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)


In [ ]:
# Train Decision Tree Classifier
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Tree depth: {model.get_depth()}")
print(f"Number of leaves: {model.get_n_leaves()}")

# Make predictions
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.3f}")


In [ ]:
# Visualize the decision tree
plt.figure(figsize=(20, 10))
plot_tree(model, feature_names=iris.feature_names, 
         class_names=iris.target_names, filled=True, rounded=True)
plt.title("Decision Tree Visualization")
plt.show()


In [ ]:
# Print tree as text
tree_rules = export_text(model, feature_names=iris.feature_names)
print("Decision Tree Rules:")
print(tree_rules)


## Validation & Testing

Let's validate our model and check for overfitting.


In [ ]:
# Validation 1: Check for overfitting
train_pred = model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, y_pred)

print("Overfitting Check:")
print(f"  Training Accuracy: {train_accuracy:.3f}")
print(f"  Test Accuracy: {test_accuracy:.3f}")
print(f"  Difference: {train_accuracy - test_accuracy:.3f}")

# Assertions
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
assert validation_result['valid'], "Invalid predictions!"
assert test_accuracy > 0.5, "Test accuracy should be better than random!"
print("\n✓ Basic validation checks passed")


In [ ]:
# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")

stability = check_cross_validation_stability(cv_scores, threshold=0.1)
print(f"  Is Stable: {stability['is_stable']}")
print("\n✓ Cross-validation checks passed")


## Performance Benchmarking

Let's benchmark performance and compare different tree depths.


In [ ]:
# Compare different max_depth values
depths = range(1, 11)
train_scores = []
test_scores = []

for depth in depths:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, dt.predict(X_train)))
    test_scores.append(accuracy_score(y_test, dt.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, 'o-', label='Training Accuracy')
plt.plot(depths, test_scores, 's-', label='Test Accuracy')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Tree Performance vs Depth')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Optimal depth appears around max_depth=3-4")


## Traceability

Let's extract feature importance and trace decision paths.


In [ ]:
# Extract feature importance
feature_importance = extract_feature_importance_trace(model, feature_names=X.columns.tolist())
print("Feature Importance:")
print(feature_importance)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_importance)), feature_importance['importance'], align='center')
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Importance')
plt.title('Feature Importance (Decision Tree)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Trace decision path for a sample
sample_idx = 0
sample = X_test.iloc[sample_idx:sample_idx+1].values[0]
path_info = trace_decision_path(model, sample, feature_names=X.columns.tolist())

print(f"Sample: {dict(zip(X.columns, sample))}")
print(f"\nDecision Path:")
for step in path_info['path']:
    print(f"  {step['feature']} <= {step['threshold']:.3f} (value: {step['value']:.3f})")
print(f"\nPrediction: {iris.target_names[int(path_info['prediction'])]}")
print(f"Actual: {iris.target_names[y_test.iloc[sample_idx]]}")


## Regression Example

Let's also apply decision trees to regression.


In [ ]:
# Regression Example: Diabetes dataset
diabetes = load_diabetes()
X_reg = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y_reg = pd.Series(diabetes.target, name='Target')

X_reg_train, X_reg_test, y_reg_train, y_reg_test = split_data(X_reg, y_reg, test_size=0.2, random_state=42)

# Train Decision Tree Regressor
dt_reg = DecisionTreeRegressor(max_depth=4, random_state=42)
dt_reg.fit(X_reg_train, y_reg_train)

y_reg_pred = dt_reg.predict(X_reg_test)
rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))

print("Decision Tree Regression:")
print(f"  RMSE: {rmse:.3f}")
print(f"  Tree depth: {dt_reg.get_depth()}")
print(f"  Number of leaves: {dt_reg.get_n_leaves()}")


## Real-World Application

Let's tune hyperparameters using GridSearchCV.


In [ ]:
# Hyperparameter tuning
param_grid = {
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Best Hyperparameters:")
print(grid_search.best_params_)
print(f"\nBest CV Accuracy: {grid_search.best_score_:.3f}")

# Train with best parameters
best_model = grid_search.best_estimator_
best_pred = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, best_pred)
print(f"Test Accuracy with Best Model: {best_accuracy:.3f}")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Decision Tree Basics**
   - Recursive binary splitting based on feature values
   - Uses entropy/Gini for classification, variance for regression
   - Creates interpretable decision rules

2. **Splitting Criteria**
   - Information Gain: Maximize reduction in entropy
   - Gini Impurity: Measure of node impurity
   - Variance Reduction: For regression tasks

3. **Overfitting Prevention**
   - Limit tree depth (max_depth)
   - Require minimum samples to split (min_samples_split)
   - Require minimum samples in leaves (min_samples_leaf)

4. **Best Practices**
   - Start with small max_depth and increase gradually
   - Use cross-validation to find optimal hyperparameters
   - Visualize trees to understand model decisions
   - Use as baseline before ensemble methods

### When to Use Decision Trees

✅ **Good for:**
- Interpretability is important
- Non-linear relationships
- Mixed data types
- Feature importance analysis
- Baseline for ensemble methods

❌ **Not ideal for:**
- High accuracy requirements (use ensembles)
- Very large datasets (computational cost)
- Linear relationships (use linear models)
- Real-time predictions (can be slow)

### Next Steps

- Try **Random Forest** (ensemble of trees) for better accuracy
- Explore **Gradient Boosting** for sequential improvement
- Consider **XGBoost** for optimized tree boosting
- Use **Pruning** techniques to reduce overfitting
